# Synthetic Population Pipeline — Results Notebook

This notebook documents all experimental outputs from the three-stage synthetic population
generation framework:

1. **Stage 1 — GAN Microdata Generation** (`Population_Synth/`)
2. **Stage 2 — Household Reconstruction** (`Household_Match/`)
3. **Stage 3 — SA Population Fitting** (`sa_run_again_n/`)

Each section shows where outputs are saved and provides **CHECK THIS** placeholders
that must be inspected to verify experimental results before revising the paper.

---
**Reviewer response map**
| Section | Reviewer concern addressed |
|---------|---------------------------|
| 1.2 | R1 & R2: GAN architecture not described; DAG relationship unclear |
| 1.3 | R1: Validation beyond RSSZ (out-of-sample statistics) |
| 1.4 | R2 App A.2: Repair/rejection rate and distribution bias |
| 2.1 | R2 P30: Algorithm is a greedy heuristic, not Hungarian |
| 3.2 | R2 P31: SA-Only baseline added |
| 3.3 | R2 Table 2: Max RSSZ outlier analysis |
---

In [ ]:
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Image

BASE = os.path.abspath('.')
POP_SYNTH  = os.path.join(BASE, 'Population_Synth')
HH_MATCH   = os.path.join(BASE, 'Household_Match')
SA_DIR     = os.path.join(BASE, 'sa_run_again_n')
print('Working directories confirmed.')

---
## 1  Stage 1 — GAN Microdata Generation (`Population_Synth/`)

### 1.1  Input data

The GAN is trained on `LINKED_DATA_dropped.csv`, produced by `step_0 fuse_to_one.py`.
This merges the 2021 Australian Census dwelling, family, and person tables for Brisbane
(area enumeration codes 31–42).

In [ ]:
orig_path = os.path.join(POP_SYNTH, 'LINKED_DATA_dropped.csv')
orig = pd.read_csv(orig_path)
print(f'Original microdata: {len(orig):,} records, {orig.shape[1]} columns')
orig.head()

### 1.2  DATGAN architecture summary

The DATGAN is trained with the following hyperparameters (edit if these changed):

| Parameter | Value |
|-----------|-------|
| Batch size | 1116 (= ~1% of dataset) |
| Epochs | 10 (initial) / 100 (production run) |
| Sample multiplier | 10 × original size |
| DAG edges | 28 (core) / 50+ (production) |
| Activation | tanh (generator), sigmoid (discriminator) |

The DAG encodes domain knowledge about variable dependencies (e.g. Age → Income,
DwellingType → Bedrooms → Vehicles). The generator is conditioned on the DAG structure
so that each synthetic variable is generated in topological order, respecting upstream
dependencies. This is distinct from a Bayesian Network: the DAG defines the *conditioning
order* for the GAN, not a full probabilistic graphical model.

In [ ]:
synth_path = os.path.join(POP_SYNTH, 'core_brisbane', 'data', 'DATGAN.csv')
synth = pd.read_csv(synth_path)
print(f'Synthetic microdata: {len(synth):,} records ({len(synth)/len(orig):.1f}× original)')
synth.head()

### 1.3  GAN validation — marginal distributions

Run `gan_validation.py` to produce these plots.  Each bar chart shows the original
census distribution (left) vs the synthetic distribution (right) for one attribute.

**CHECK THESE PLOTS** — for attributes *not* used in the SA fitting (marked
`out-of-sample` in `validation_summary.csv`), do the distributions match the original?
Large discrepancies indicate the GAN has not learned those relationships.

In [ ]:
val_dir = os.path.join(POP_SYNTH, 'gan_validation_outputs')
summary_csv = os.path.join(val_dir, 'validation_summary.csv')
if os.path.exists(summary_csv):
    summary = pd.read_csv(summary_csv)
    print('Columns with highest divergence (worst GAN fit):')
    display(summary.sort_values('value', ascending=False).head(15))
else:
    print('validation_summary.csv not found — run gan_validation.py first.')

In [ ]:
# Show marginal distribution plots (first 6 columns)
marginal_dir = os.path.join(val_dir, 'marginal_distributions')
plots = sorted(glob.glob(os.path.join(marginal_dir, '*.png')))[:6]
if plots:
    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    for ax, p in zip(axes.flat, plots):
        img = mpimg.imread(p)
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(os.path.basename(p).replace('.png', ''), fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print('No marginal distribution plots found — run gan_validation.py first.')

### 1.4  GAN repair / rejection rate

The post-processing filters reject or repair synthetic records that violate hard constraints
(e.g. children under 15 listed as employed).  A high repair rate indicates that the GAN's
learned joint distribution diverges from the constraint space, which could bias the final
population (Reviewer 2, Appendix A.2).

**CHECK THIS VALUE** — if repair rate > 5%, discuss the potential bias in the paper.

In [ ]:
repair_report = os.path.join(val_dir, 'repair_report.txt')
if os.path.exists(repair_report):
    with open(repair_report) as f:
        print(f.read())
else:
    print('repair_report.txt not found — run gan_validation.py first.')

---
## 2  Stage 2 — Household Reconstruction (`Household_Match/`)

### 2.1  Algorithm clarification

> **Paper correction (Reviewer 2, P30 / Algorithm 1)**
> The paper described Algorithm 1 as 'Hungarian Algorithm Matching'.  This was incorrect.
> The household formation step uses a **greedy sequential heuristic**: individuals are
> grouped by shared structural attributes (area, household size, family count) and assigned
> in sequential order.  This runs in O(n) and is practical for populations of millions,
> unlike the Hungarian algorithm which is O(n³).
>
> The greedy assignment is then refined by **Simulated Annealing** (delta-scoring),
> which is the actual optimisation step.  The code in `houehold_family_match_c.py` has
> been updated with accurate docstrings.

**Tradeoff of greedy vs optimal assignment**: The greedy heuristic may produce a lower
initial compatibility score than an optimal Hungarian assignment, but the SA refinement
compensates.  For future work, a partition-then-match strategy could apply Hungarian
matching within small compatibility groups to improve initialisation quality.

In [ ]:
hh_path = os.path.join(HH_MATCH, 'cleaned_dataset_hhid.csv')
if os.path.exists(hh_path):
    hh_df = pd.read_csv(hh_path)
    print(f'Household-matched population: {len(hh_df):,} records')
    print(f'Unique households: {hh_df["household_id"].nunique():,}')
    display(hh_df.head())
else:
    print('cleaned_dataset_hhid.csv not found — run Stage 2 first.')

---
## 3  Stage 3 — SA Population Fitting (`sa_run_again_n/`)

### 3.1  SA-GAN results (main experiment)

The SA-GAN experiment applies Simulated Annealing to the GAN-enriched microdata pool
to select households for each SA2 zone to match the census control tables.
Results are in `sa_run_again_n/zones_sa_*/`.

**CHECK THIS TABLE** — compare with Table 2 of the paper.

In [ ]:
# Load any existing per-zone RSSZ results from the completed SA runs
result_dirs = sorted(glob.glob(os.path.join(SA_DIR, 'zones_sa_*')))
rssz_files = []
for d in result_dirs:
    rssz_files += glob.glob(os.path.join(d, '*.csv'))
print(f'Found {len(rssz_files)} zone result files across {len(result_dirs)} directories.')

# Summarise if rssz_results directory exists
rssz_result_dir = os.path.join(SA_DIR, 'rssz_results')
if os.path.exists(rssz_result_dir):
    rssz_csvs = sorted(glob.glob(os.path.join(rssz_result_dir, '*.csv')))
    if rssz_csvs:
        combined = pd.concat([pd.read_csv(f) for f in rssz_csvs], ignore_index=True)
        print('\nRSSZ summary (SA-GAN):')
        print(combined.describe())
else:
    print('rssz_results directory not found.')

### 3.2  SA-Only baseline (new experiment)

**Reviewer 2 (P31)**: SA is applied directly to the original census microdata *without* GAN
expansion.  This isolates the contribution of SA optimisation from the GAN microdata pool.

Run: `python sa_run_again_n/sa_only_baseline.py`

**CHECK THIS TABLE** — add the SA-Only row to Table 2 of the paper and discuss.

In [ ]:
sa_only_summary = os.path.join(SA_DIR, 'sa_only_results', 'sa_only_rssz_summary.csv')
if os.path.exists(sa_only_summary):
    sa_only = pd.read_csv(sa_only_summary)
    print('SA-Only RSSZ statistics:')
    print(f'  Mean  : {sa_only["rssz_total"].mean():.4f}')
    print(f'  Median: {sa_only["rssz_total"].median():.4f}')
    print(f'  Min   : {sa_only["rssz_total"].min():.4f}')
    print(f'  Max   : {sa_only["rssz_total"].max():.4f}')
    print(f'  Std   : {sa_only["rssz_total"].std():.4f}')
    display(sa_only.head(20))
else:
    print('sa_only_rssz_summary.csv not found — run sa_only_baseline.py first.')

### 3.3  RSSZ outlier analysis

**Reviewer 2 (Table 2)**: The max RSSZ of 7.52 for SA-GAN must be explained.
Run `rssz_outlier_analysis.py` to identify which zones and control tables drive the outliers.

**CHECK THIS PLOT** — inspect `rssz_outlier_outputs/rssz_distribution.png`.

In [ ]:
outlier_dir = os.path.join(SA_DIR, 'rssz_outlier_outputs')
dist_plot = os.path.join(outlier_dir, 'rssz_distribution.png')
if os.path.exists(dist_plot):
    display(Image(dist_plot))
else:
    print('rssz_distribution.png not found — run rssz_outlier_analysis.py first.')

In [ ]:
outlier_csv = os.path.join(outlier_dir, 'outlier_zones.csv')
driver_csv = os.path.join(outlier_dir, 'driver_tables.csv')
if os.path.exists(outlier_csv):
    print('Outlier zones:')
    display(pd.read_csv(outlier_csv))
if os.path.exists(driver_csv):
    print('\nControl tables most frequently causing high RSSZ:')
    display(pd.read_csv(driver_csv))
else:
    print('Outlier analysis files not found — run rssz_outlier_analysis.py first.')

In [ ]:
# Show per-zone breakdown plots for each outlier
outlier_plots = sorted(glob.glob(os.path.join(outlier_dir, 'outlier_zone_*.png')))
for p in outlier_plots:
    print(os.path.basename(p))
    display(Image(p))

---
## 4  Experiment comparison table

Reconstruct Table 2 once all experiments have run.

**CHECK THIS TABLE** — fill in the SA-Only row from the output above.
Compare with values in the paper for IPU-Only, IPU-GAN, SA-GAN.

In [ ]:
# Placeholder: update with actual values once experiments complete
table2 = pd.DataFrame([
    {'Method': 'IPU-Only',  'Mean RSSZ': '— CHECK PAPER TABLE 2 —', 'Max RSSZ': '—', 'Std': '—'},
    {'Method': 'IPU-GAN',   'Mean RSSZ': '— CHECK PAPER TABLE 2 —', 'Max RSSZ': '—', 'Std': '—'},
    {'Method': 'SA-Only',   'Mean RSSZ': '— RUN sa_only_baseline.py —', 'Max RSSZ': '—', 'Std': '—'},
    {'Method': 'SA-GAN',    'Mean RSSZ': '— CHECK PAPER TABLE 2 —', 'Max RSSZ': '—', 'Std': '—'},
])
display(table2)

---
## 5  Joint distribution comparison (out-of-sample validation)

**Reviewer 1**: validation using statistics not in the fitting process.

The `gan_validation.py` script generates joint frequency heatmaps for attribute pairs
that were NOT used as control tables during SA fitting.  If these match the original
census distributions, the GAN has learned realistic joint relationships.

**CHECK THESE PLOTS** — large differences indicate the GAN or SA is distorting
multi-attribute relationships.

In [ ]:
joint_dir = os.path.join(POP_SYNTH, 'gan_validation_outputs', 'joint_distributions')
joint_plots = sorted(glob.glob(os.path.join(joint_dir, '*.png')))
for p in joint_plots:
    print(os.path.basename(p))
    display(Image(p))

---
## 6  Runtime summary

**Reviewer 1**: 'A more explicit discussion of scalability and runtime trade-offs would be helpful.'

Fill in the table below after running the full pipeline.

In [ ]:
runtime_table = pd.DataFrame([
    {'Stage': 'Data fusion (step_0)',       'Runtime': '— CHECK run_pipeline.py output —'},
    {'Stage': 'GAN training (10 epochs)',   'Runtime': '— CHECK run_pipeline.py output —'},
    {'Stage': 'GAN sampling',               'Runtime': '— CHECK run_pipeline.py output —'},
    {'Stage': 'Household assignment + SA',  'Runtime': '— CHECK run_pipeline.py output —'},
    {'Stage': 'Household repair',           'Runtime': '— CHECK run_pipeline.py output —'},
    {'Stage': 'SA fitting (per zone)',       'Runtime': '— CHECK run_pipeline.py output —'},
    {'Stage': 'SA-Only baseline (per zone)','Runtime': '— CHECK run_pipeline.py output —'},
])
display(runtime_table)